# YOLO11 on VisDrone - CMPE 401 Project 1
Runtime -> Change runtime type -> **T4 GPU**, then **Run all**. About 35-45 min. Results land in `results/`, `figures/`, `README.md`.

In [ ]:
!pip -q install ultralytics
import os; os.makedirs('src', exist_ok=True); os.makedirs('results', exist_ok=True)

In [ ]:
%%writefile src/run_experiments.py
"""Train + evaluate every experiment on VisDrone-DET with Ultralytics YOLO.
Usage:  python src/run_experiments.py            (all runs)
        python src/run_experiments.py baseline   (one run)
Budget knobs are at the top: EPOCHS / FRACTION trade accuracy for wall-clock time."""
import sys, json, time
from pathlib import Path
from ultralytics import YOLO

EPOCHS, FRACTION, BATCH, SEED = 12, 0.2, 16, 0   # FRACTION = share of the 6,471 train images used
DATA = "VisDrone.yaml"                            # Ultralytics auto-downloads VisDrone2019-DET on first use
OUT = Path("runs"); RES = Path("results"); RES.mkdir(exist_ok=True)

COMMON = dict(data=DATA, epochs=EPOCHS, fraction=FRACTION, batch=BATCH, seed=SEED, deterministic=True,
              imgsz=640, patience=EPOCHS, plots=True, project=str(OUT), exist_ok=True, workers=2, verbose=False)

# Each experiment differs from `baseline` in exactly the keys listed here (controlled variables).
EXPERIMENTS = {
    "baseline":   dict(model="yolo11n.pt"),                                        # Part I
    "size_s":     dict(model="yolo11s.pt"),                                        # Part III: model capacity n -> s
    "regularized": dict(model="yolo11n.pt", cos_lr=True, weight_decay=0.002,       # Part IV: regularisation cycle
                        scale=0.7, mixup=0.1, hsv_s=0.8, fliplr=0.5),
    "yolov8n":    dict(model="yolov8n.pt"),                                        # Part V: other YOLO version
}

def run(name):
    cfg = dict(COMMON); cfg.update(EXPERIMENTS[name]); weights = cfg.pop("model")
    model = YOLO(weights)
    t0 = time.time(); model.train(name=name, **cfg); train_s = time.time() - t0
    best = YOLO(OUT / name / "weights" / "best.pt")
    m = best.val(data=DATA, imgsz=cfg["imgsz"], split="val", plots=True, project=str(OUT), name=f"{name}_val", exist_ok=True, verbose=False)
    out = dict(name=name, weights=weights, overrides={k: v for k, v in EXPERIMENTS[name].items() if k != "model"},
               params_M=round(sum(p.numel() for p in best.model.parameters()) / 1e6, 2),
               precision=float(m.box.mp), recall=float(m.box.mr), map50=float(m.box.map50), map50_95=float(m.box.map),
               infer_ms=round(m.speed["inference"], 2), train_minutes=round(train_s / 60, 1),
               epochs=EPOCHS, fraction=FRACTION)
    (RES / f"{name}.json").write_text(json.dumps(out, indent=2)); print("RESULT", out)

if __name__ == "__main__":
    for n in (sys.argv[1:] or EXPERIMENTS): run(n)


In [ ]:
%%writefile src/analyze.py
"""Turn runs/*/results.csv + results/*.json into figures, tables and an auto-written analysis (results/RESULTS.md)."""
import json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

RES, FIG, RUNS = Path("results"), Path("figures"), Path("runs"); FIG.mkdir(exist_ok=True)
ORDER = ["baseline", "size_s", "regularized", "yolov8n"]
J = {n: json.loads((RES / f"{n}.json").read_text()) for n in ORDER if (RES / f"{n}.json").exists()}

def curves(n):
    d = pd.read_csv(RUNS / n / "results.csv"); d.columns = [c.strip() for c in d.columns]
    d["train_loss"] = d["train/box_loss"] + d["train/cls_loss"] + d["train/dfl_loss"]
    d["val_loss"] = d["val/box_loss"] + d["val/cls_loss"] + d["val/dfl_loss"]
    return d

def diagnose(d):
    """Rule-based fit diagnosis from the last 5 epochs."""
    k = min(5, len(d)); e = np.arange(k)
    sv = np.polyfit(e, d["val_loss"].tail(k), 1)[0]; st = np.polyfit(e, d["train_loss"].tail(k), 1)[0]
    sm = np.polyfit(e, d["metrics/mAP50(B)"].tail(k), 1)[0]
    gap = d["val_loss"].iloc[-1] / d["train_loss"].iloc[-1]
    if sv > 0 and st < 0: tag = "overfitting (validation loss rising while training loss falls)"
    elif sv < 0 and sm > 0.002: tag = "underfitting / not yet converged (validation loss and mAP50 still improving)"
    else: tag = "approximately converged (validation loss flat)"
    return tag, gap, sv, st, sm

# --- Part I/II: baseline curves -------------------------------------------------------------
b = curves("baseline")
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (t, v, title) in zip(ax, [("train/box_loss", "val/box_loss", "Box loss"), ("train/cls_loss", "val/cls_loss", "Class loss"), ("train_loss", "val_loss", "Total loss")]):
    a.plot(b["epoch"], b[t], label="train"); a.plot(b["epoch"], b[v], label="val"); a.set_title(title); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "baseline_loss_curves.png", dpi=150); plt.close()
fig, ax = plt.subplots(figsize=(6, 4))
for c, l in [("metrics/mAP50(B)", "mAP50"), ("metrics/mAP50-95(B)", "mAP50-95"), ("metrics/precision(B)", "precision"), ("metrics/recall(B)", "recall")]: ax.plot(b["epoch"], b[c], label=l)
ax.set_xlabel("epoch"); ax.set_title("Baseline validation metrics"); ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.savefig(FIG / "baseline_metrics.png", dpi=150); plt.close()

import shutil
for src, dst in [("baseline_val/confusion_matrix_normalized.png", "baseline_confusion_matrix.png"), ("baseline_val/BoxPR_curve.png", "baseline_pr_curve.png")]:
    if (RUNS / src).exists(): shutil.copy(RUNS / src, FIG / dst)

# --- comparison plots -------------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n in J:
    d = curves(n); ax[0].plot(d["epoch"], d["val_loss"], label=n); ax[1].plot(d["epoch"], d["metrics/mAP50(B)"], label=n)
ax[0].set_title("Validation total loss"); ax[1].set_title("Validation mAP50")
for a in ax: a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "comparison.png", dpi=150); plt.close()

# --- tables + auto-written analysis ------------------------------------------------------------
def table(names):
    rows = ["| Run | Weights | Params (M) | Precision | Recall | mAP50 | mAP50-95 | Infer ms/img | Train min |", "|---|---|---|---|---|---|---|---|---|"]
    for n in names:
        if n in J: r = J[n]; rows.append(f"| {n} | {r['weights']} | {r['params_M']} | {r['precision']:.3f} | {r['recall']:.3f} | {r['map50']:.3f} | {r['map50_95']:.3f} | {r['infer_ms']} | {r['train_minutes']} |")
    return "\n".join(rows)

tag, gap, sv, st, sm = diagnose(b); r0 = J["baseline"]
md = [f"## Results (YOLO11, VisDrone-DET, {r0['fraction']*100:.0f}% of train images, {r0['epochs']} epochs, imgsz 640, seed 0)\n",
      "### Part I - Baseline\n", table(["baseline"]), "\n![baseline loss](figures/baseline_loss_curves.png)\n![baseline metrics](figures/baseline_metrics.png)\n![confusion matrix](figures/baseline_confusion_matrix.png)\n![PR curve](figures/baseline_pr_curve.png)\n",
      "### Part II - Loss-curve and fitting analysis\n",
      f"Final-epoch train total loss = {b['train_loss'].iloc[-1]:.3f}, validation total loss = {b['val_loss'].iloc[-1]:.3f} (val/train ratio {gap:.2f}). "
      f"Over the last {min(5,len(b))} epochs the validation-loss slope is {sv:+.4f}/epoch, training-loss slope {st:+.4f}/epoch, and mAP50 slope {sm:+.4f}/epoch. "
      f"**Diagnosis: {tag}.**\n",
      f"*Dataset size.* Training uses {r0['fraction']*100:.0f}% of VisDrone-DET (about {int(6471*r0['fraction'])} images) with many tiny, densely packed objects; a small training set lets a model memorise appearance and layout, which raises the val/train gap. "
      f"*Model capacity.* YOLO11n has {r0['params_M']} M parameters; if capacity were the bottleneck, both losses would stay high together (underfitting) and a larger model would help - tested in the size experiment below.\n"]
if "size_s" in J:
    s = J["size_s"]; d = s["map50"] - r0["map50"]
    md += ["### Part III - Controlled experiment: model capacity (only variable: n -> s)\n", table(["baseline", "size_s"]),
           f"\nChanging YOLO11n to YOLO11s ({r0['params_M']} M -> {s['params_M']} M parameters) changed mAP50 by {d:+.3f} ({r0['map50']:.3f} -> {s['map50']:.3f}) and inference time from {r0['infer_ms']} to {s['infer_ms']} ms/img. "
           + ("The gain indicates the baseline is capacity-limited on small objects, at the cost of speed." if d > 0.005 else "No clear gain indicates the baseline is limited by data/training budget rather than capacity.") + "\n"]
if "regularized" in J:
    g = J["regularized"]; d = g["map50"] - r0["map50"]; tg, gg, *_ = diagnose(curves("regularized"))
    md += ["### Part IV - Improvement cycle: regularisation\n",
           f"**Baseline** (above) -> **Experimental settings** identical except -> **Controlled modification**: `{g['overrides']}` (cosine LR decay, 4x weight decay, stronger scale/colour augmentation, mixup 0.1). "
           "Rationale: weight decay and augmentation reduce variance (overfitting) and cosine decay lets the optimiser settle into a flatter minimum at the end of a short schedule.\n", table(["baseline", "regularized"]),
           f"\n**Evaluation / analysis**: mAP50 {r0['map50']:.3f} -> {g['map50']:.3f} ({d:+.3f}); val/train loss ratio {gap:.2f} -> {gg:.2f}; diagnosis now: {tg}. "
           + ("**Conclusion**: the regularisation helped and is kept." if d > 0 else "**Conclusion**: with only a short schedule the added regularisation slows learning more than it prevents overfitting, so it did not help here; with a longer schedule it is expected to pay off.") + "\n"]
if "yolov8n" in J:
    y = J["yolov8n"]
    md += ["### Part V - Multi-version comparison (YOLO11n vs YOLOv8n)\n", table(["baseline", "yolov8n"]),
           f"\nYOLOv8n reaches mAP50 {y['map50']:.3f} vs {r0['map50']:.3f} for YOLO11n, with {y['params_M']} M vs {r0['params_M']} M parameters and {y['infer_ms']} vs {r0['infer_ms']} ms/img inference.\n"]
md += ["### All runs\n", table(ORDER), "\n![comparison](figures/comparison.png)\n"]
(RES / "RESULTS.md").write_text("\n".join(md)); print("\n".join(md))


In [ ]:
%%writefile src/build_readme.py
"""README.md = README_template.md + results/RESULTS.md  (run after analyze.py)"""
from pathlib import Path
t = Path("README_template.md").read_text(); r = Path("results/RESULTS.md").read_text()
Path("README.md").write_text(t.replace("<!--RESULTS-->", r)); print("README.md written")


In [ ]:
%%writefile README_template.md
# YOLO11 on VisDrone-DET: training dynamics, controlled experiments and multi-version comparison

CMPE 401 Instructor-defined Project 1. This repo trains a modern YOLO detector on VisDrone-DET (small, dense, real-world drone objects), analyses its loss curves, runs a controlled experiment and a regularisation improvement cycle, and compares YOLO11 against YOLOv8. Every number below is produced by the code in `src/`; nothing is typed by hand.

## 1. Objective
Understand the YOLO detection pipeline, fine-tune YOLO11, diagnose over/underfitting from training vs validation loss, and quantify the effect of model size, regularisation and YOLO version.

## 2. Setup
| Item | Value |
|---|---|
| Dataset | VisDrone2019-DET (10 classes, 6,471 train / 548 val images), downloaded automatically by Ultralytics (`VisDrone.yaml`) |
| Framework | Ultralytics (YOLO11 / YOLOv8), PyTorch |
| Training budget | 20% of train images (`fraction=0.2`), 12 epochs, batch 16, imgsz 640, seed 0, single T4 GPU (Colab) |
| Metrics | Precision, Recall, mAP50, mAP50-95 on the full validation set (`best.pt`), inference ms/img |
| Why a reduced budget | The whole study had to fit in a few GPU-minutes per run. Absolute mAP is therefore lower than a full 100+ epoch run; all comparisons use the same budget, so relative conclusions are fair. Raise `EPOCHS`/`FRACTION` in `src/run_experiments.py` for the full run. |

## 3. How the pipeline works
1. `VisDrone.yaml` downloads the images and converts VisDrone annotations to YOLO format (class, x, y, w, h normalised).
2. YOLO11 = backbone (C3k2 blocks + SPPF + C2PSA attention) -> neck (multi-scale feature fusion, P3/P4/P5) -> decoupled anchor-free head. Small objects are detected on the high-resolution P3 map (stride 8).
3. Loss = box loss (CIoU) + class loss (BCE) + DFL (distribution focal loss for box edges). We report their sum as "total loss".
4. `src/run_experiments.py` trains each configuration; `src/analyze.py` plots curves, builds tables and writes the analysis; `src/build_readme.py` assembles this README.

## 4. Experiments
| Run | Change from baseline (everything else identical) | Purpose |
|---|---|---|
| `baseline` | YOLO11n, imgsz 640 | Part I |
| `size_s` | YOLO11s (more parameters) | Part III: capacity |
| `regularized` | cosine LR, weight decay 0.002 (4x), scale 0.7, mixup 0.1, hsv_s 0.8 | Part IV: overfitting control |
| `yolov8n` | YOLOv8n | Part V: version comparison |

<!--RESULTS-->

## 5. Reproduce
```bash
pip install ultralytics
python src/run_experiments.py      # trains all 4 runs (~6-10 min each on a T4)
python src/analyze.py              # figures + results/RESULTS.md
python src/build_readme.py         # rebuilds this README
```
Or open `notebooks/yolo11_visdrone_colab.ipynb` in Colab (Runtime -> T4 GPU -> Run all).

## 6. Limitations and next steps
Reduced training budget (20% data, 12 epochs); single seed, so differences under ~0.005 mAP50 are within noise; the optional VisDrone test-challenge submission was not run. Next: full data, 100 epochs, imgsz 960 for small objects, multiple seeds.


## Train all experiments (baseline, size, regularisation, YOLOv8n)
First run downloads VisDrone (~2 GB).

In [ ]:
!python src/run_experiments.py

## Analyse + build README

In [ ]:
!python src/analyze.py && python src/build_readme.py

In [ ]:
from IPython.display import Image, display
for f in ['baseline_loss_curves','baseline_metrics','comparison']: display(Image(f'figures/{f}.png'))

In [ ]:
!zip -qr yolo_results.zip README.md results figures src runs/*/results.csv
from google.colab import files; files.download('yolo_results.zip')